In [1]:
from pprint import pprint
import pandas as pd
from pathlib import Path
import msgspec
import numpy as np

encoder = msgspec.json.Encoder()
decoder = msgspec.json.Decoder()


def read_json(file_path, jsonl=False):
    file_path = Path(file_path)
    if not file_path.is_file():
        raise ValueError("filepath is not a file")
    # if not file_path.suffix == ".jsonl" and jsonl:
    #     raise ValueError("the file is not jsonl")
    # if not file_path.suffix == ".json" and not jsonl:
    #     raise ValueError("the file is not json")
    file_path = file_path.__str__()
    
    output = []

    print("Reading the json file...")
    with open(file_path, "rb") as file:
        data = file.read()

        if jsonl:
            output = decoder.decode_lines(data)
        else:
            output = decoder.decode(data)

    print(f"The file is of type: {type(output)}")
    print(f"The file contains {len(output)} items.")
    return output


def write_json(file_path, data, jsonl=False):
    with open(file_path, "wb") as file:
        if jsonl:
            file.write(encoder.encode_lines(data))
        else:
            file.write(encoder.encode(data))
    print(f"The file contains {len(data)} items.")
    print("Saved to", file_path)

def read_tsv(file_path, row_names=None):
    file_path = Path(file_path)
    if file_path.is_file() and file_path.suffix == ".tsv" :
        temp = pd.read_csv(file_path, sep='\t', names=row_names)
    else:
        raise ValueError("filepath is not a file or it is not a tsv file.")
    return temp


def read_txt(file_path, lines=False):
    with open(file_path, "r") as f:
        if lines:
            return f.readlines()
        return f.read()


# Refine qrel

In [62]:
import os

# export VLLM_USE_V1=1
# export TOKENIZERS_PARALLELISM=0
os.environ["VLLM_USE_V1"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "0"


from vllm import LLM, SamplingParams
from vllm.distributed import cleanup_dist_env_and_memory

# sampling_params = SamplingParams(
#     max_tokens=16, 
#     temperature=0.7,
#     top_p=0.8,
#     top_k=20,
#     min_p=0
# )

# llm = LLM(
#     model="Qwen/Qwen3-4B-Instruct-2507",
#     enable_chunked_prefill=True,
#     enable_prefix_caching=True,
#     generation_config="auto",
#     max_model_len=32768,  # Limit context window
#     max_num_seqs=4,  # Limit batch size
#     gpu_memory_utilization=0.95,
#     disable_cascade_attn=True,  # Avoid gibberish output due to batch inference
#     seed=42,
# )

In [3]:
df = pd.read_parquet("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/test/corpus23009_sentencechunk128_with_test_query.parquet")

In [4]:
# prompt_template = """Given an input with the following format:
# (index, query, list of answer_terms, passage)

# Your task is to decide whether the passage is relevant to the query for qrel construction. More specifically, you need to check if the passage is relevant not only in terms of containing the answer_terms, but also satisfying the temporal constraints expressed in the query, so that they can be used for a time-sensitive question answering task.

# A passage is relevant if and only if BOTH of the following are true:

# 1. Most important criteria: The passage must strictly satisfy the temporal constraint(s) expressed in the query, i.e., the passage must contain the correct time period or event as indicated by the query. (e.g., in, between, before, after, from ... to ..., etc.)
# 2. The passage must contain all the answer_terms, and the temporal information must be associated with the answer_terms in the passage (i.e., the answer_terms must be valid within the specified time frame).
# 3. If the passage does not satisfy both 1 and 2, the passage is considered not relevant.

# Output format:
# - You must output the integer index only if the passage is relevant.
# - Otherwise you must output only "-1".
# - You must not output explanations or additional text.

prompt_template = """
Input format
(index, query, list of answer_terms, passage)

Task
Decide whether the passage is relevant to the query for temporal information retrieval (qrel construction).
A passage is relevant if and only if ALL of the following conditions are satisfied:
1. Temporal correctness (mandatory): The passage must clearly satisfy the temporal constraint(s) explicitly or implicitly expressed in the query (e.g., in, before, after, between, from … to …).
- Explicit dates or time spans in the passage are acceptable.
- Implicit temporal coverage is acceptable only if it unambiguously covers the queried time period (e.g., “served from 1989–1993” satisfies “in 1991”).
- Vague, approximate, or underspecified temporal expressions (e.g., “around that time”, “in the early days”, “in the 1900s”) do not satisfy temporal constraints.
- If the temporal alignment between the query and passage is unclear, the passage must be judged not relevant.
2. Answer-term validity within time: The passage must contain all answer_terms.
- The temporal information must be directly associated with the answer_terms, meaning the answer_terms refer to the same entity, event, or state that is valid within the specified time frame.
- Merely mentioning the answer_terms elsewhere in the passage without temporal grounding is insufficient.
3. No external inference: Judge relevance only based on the information explicitly present in the passage.
- Do not rely on external knowledge, common facts, or assumptions.
- Do not infer missing dates or temporal relations.
4. Binary decision rule: If any of the above conditions is not satisfied, the passage is not relevant.

Output format
- Output the integer index only if the passage is relevant.
- Otherwise, output -1.
- Output only the index or -1.
- Do not include explanations, reasoning steps, or additional text.
- Do not guess or assign partial relevance.

### Demonstration 1
Input:
(Index: 123, Question: Who was the head coach of the team 1. FC Köln from Nov 2019 to Nov 2020?, Answers: ['Markus Gisdol'], Passage: "'Section: Decline and changes (2018–). FC Saarbrücken, the club decided to terminate Beierlorzer's contract on 9 November 2019. Sporting director Armin Veh, who weeks earlier had announced that he would not extend his contract with the club, was also dismissed from his position. On 18 November, former HSV manager Markus Gisdol was appointed to the club's head coaching position, while Horst Heldt was made sporting director.  Both signed contracts until 2021. After avoiding relegation at the end of the season, Gisdol's contract was extended until 2023. )"
Output: 123
### End of Demonstration 1

### Demonstration 2
Input:
(Index: 2, Question: Who was the head coach of the team 1. FC Köln from Nov 2019 to Nov 2020?, Answers: ['Markus Gisdol'], Passage: "Section: Decline and changes (2018–).   The club found itself in a renewed relegation during the 2020–21 season. On 11 April 2021, after losing to relegation rival Mainz 05, Gisdol was dismissed from his position as head coach. The next day, it was announced that Friedhelm Funkel would take over head coaching duties until the end of the season. On 11 May, it was reported that SC Paderborn manager Steffen Baumgart would succeed Funkel as head coach at the beginning of the 2021–22 season. )"
Output: -1
### End of Demonstration 2

### Demonstration 3
Input:
(Index: 3, Question: Who was the head coach of 1. FC Köln from Nov 2019 to Nov 2020?, Answers: ['Markus Gisdol'], Passage: "'Gisdol was appointed head coach in November 2018 and left in October 2019 before the season ended.'")
Output: -1
### End of Demonstration 3
"""

In [ ]:
# row = df.iloc[0]
# index = 0
# question = row['question']
# answers = row['targets'][0] if len(row['targets']) == 1 else ", ".join([x for x in row['targets']])
# passage = "Title: 1._FC_Köln. Section: Decline and changes (2018–). FC Saarbrücken, the club decided to terminate Beierlorzer's contract on 9 November 2019. Sporting director Armin Veh, who weeks earlier had announced that he would not extend his contract with the club, was also dismissed from his position. On 18 November, former HSV manager Markus Gisdol was appointed to the club's head coaching position, while Horst Heldt was made sporting director.  Both signed contracts until 2021. After avoiding relegation at the end of the season, Gisdol's contract was extended until 2023." # row['text']

# messages = []

# content = f'Input: (Index: {index}, Question: "{question}", Answers: "[{answers}]", Passage: "{passage}")\nOutput:'
    
# messages.append([
#     {"role":"system", "content":prompt_template},
#     {"role":"user", "content":{content},}
# ])

In [23]:
from torch.utils.data import Dataset, DataLoader

class TemporalQrelDataset(Dataset):
    """
    Dataset for temporal qrel annotation.
    Each item is a dict with 'system' and 'user' messages.
    """
    def __init__(self, df, prompt_template):
        """
        df: DataFrame with columns 'question', 'targets', 'text'
        prompt_template: string, system prompt
        """
        self.df = df
        self.prompt_template = prompt_template

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        question = row['question']
        # targets can be a single string or list
        targets = row['targets']
        if isinstance(targets, list):
            answers = ", ".join([str(x) for x in targets])
        else:
            answers = str(targets)

        passage = row['text']

        index = idx

        user_content = f'Input: (Index: {index}, Question: "{question}", Answers: "{answers}", Passage: "{passage}")\nOutput:'

        return targets, passage, self.prompt_template, user_content

# # --------------------------------
# # Collate function for batching
# # --------------------------------
# def collate_fn(batch):
#     """
#     Converts a list of dicts into a batch suitable for LLM input.
#     Returns a dict with:
#         - 'system_messages': list of system messages
#         - 'user_messages': list of user messages
#     """
#     answers = []
#     print(batch[0])
#     for a in batch[0]:
#         print(a)
#         answers.append([x.lower() for x in a])
        

#     return answers, [x.lower() for x in batch[1]], [
#         [{"role":"system", "content":x[2]}, {"role":"user", "content":x[3]}]
#         for x in batch
#     ]

def collate_fn(batch):
    """
    Batch is a list of tuples: (targets, passage, system_message, user_content)
    Returns:
        - answers: list of lists (lowercased)
        - passages: list of passages (lowercased)
        - messages: list of system+user messages formatted for LLM
    """
    answers = []
    passages = []
    messages = []

    for item in batch:
        targets, passage, system_msg, user_content = item

        # make targets a list of lowercased strings
        if isinstance(targets, np.ndarray) and len(targets) > 1:
            ans_list = [x.lower() for x in targets]
        else:
            ans_list = [targets[0].lower()]

        answers.append(ans_list)
        passages.append(passage.lower())
        messages.append([
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_content}
        ])

    return answers, passages, messages

# df = pd.read_parquet("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/test/corpus23009_sentencechunk128_with_test_query.parquet")
# corpus = pd.read_parquet("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/test/corpus.parquet")
# df = df.merge(corpus[['docid', 'text']], on="docid")
# df.rename(columns={"text_y":"text"}, inplace=True)
# df.drop(columns=["text_x"], inplace=True)
# --------------------------------
# Example usage
# --------------------------------
# df = your DataFrame with 'question', 'targets', 'text'
dataset = TemporalQrelDataset(df, prompt_template)
dataloader = DataLoader(dataset, batch_size=2048, shuffle=False, collate_fn=collate_fn)

In [6]:
# df = pd.read_parquet("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/test/corpus23009_sentencechunk128_with_test_query.parquet")
corpus_parquet = pd.read_parquet("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/test/corpus.parquet")
query_parquet = pd.read_parquet("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/test/query.parquet")
# df = df.merge(corpus_parquet[['docid', 'text']], on="docid")
# df.rename(columns={"text_y":"text"}, inplace=True)
# df.drop(columns=["text_x"], inplace=True)

In [25]:
from tqdm.auto import tqdm
from itertools import product

result = []

for ix, (answer, passage, batch) in enumerate(tqdm(dataloader)):
    output = llm.chat(
        messages=batch, 
        sampling_params=sampling_params,
        # chat_template_kwargs={"enable_thinking": True},
    )
    for out, a, p, b in zip(output, answer, passage, batch):
        out = out.outputs[0].text
        if int(out) != -1:
            # pprint(out)
            # pprint(list(product(a, [p])))
            for x, y in product(a, [p]):
                # answered = False
                if x in y:
                    result.append(int(out))
                    # answered = True
                    break
                # if answered:
                #     break 

  0%|          | 0/59 [00:00<?, ?it/s]

Adding requests:   0%|          | 0/2048 [00:00<?, ?it/s]

Processed prompts:   0%|                                                                                      …

Adding requests:   0%|          | 0/2048 [00:00<?, ?it/s]

Processed prompts:   0%|                                                                                      …

...

Processed prompts:   0%|                                                                                      …

Adding requests:   0%|          | 0/2048 [00:00<?, ?it/s]

Processed prompts:   0%|                                                                                      …

Adding requests:   0%|          | 0/1344 [00:00<?, ?it/s]

Processed prompts:   0%|                                                                                      …

In [26]:
len(result)

5310

In [12]:
question_to_id_dict = {}
for question in df['question'].unique():
    qid = int(query_parquet[query_parquet['query'] == question]['query_id'].values[0])
    question_to_id_dict[question] = qid

In [15]:
question_to_id_dict[df.iloc[96]['question']]

5

In [27]:
from tqdm.auto import tqdm
qrel2 = []
wrong = []
for index in tqdm(result): 
    row = df.iloc[index]
    query, targets, docid, wiki_id = row['question'], row['targets'], row['docid'], row['wiki_id']
    
    for a in targets:
        check = False
        if a.lower() in corpus_parquet[corpus_parquet['docid'] == docid]['text'].values[0].lower():
            check = True
            qid = question_to_id_dict[query]
            qrel2.append(f"{qid} 0 {docid} 1")
            break
        if check:
            check = False
            break
    
    if not check:
        # print(query)
        # print(targets)
        # print(corpus_parquet.iloc[docid]['text'])
        wrong.append(index)

  0%|          | 0/5310 [00:00<?, ?it/s]

In [37]:
query_parquet[query_parquet['query_id'] == 5163]

,query,answers,query_id
121485,What was the working location for Gottfried Wi...,[Wolfenbüttel],5163


In [35]:
# corpus_parquet[corpus_parquet['docid'] == 38790]
corpus_parquet[corpus_parquet['wiki_id'] == "Q9047"]

,wiki_id,text,docid
38663,Q9047,Gottfried Wilhelm Leibniz Gottfried Wilhelm (...,38663
38664,Q9047,He also contributed to the field of library sc...,38664
38665,Q9047,"As a mathematician, his greatest achievement w...",38665
38666,Q9047,While working on adding automatic multiplicati...,38666
38667,Q9047,"Leibniz has been called the ""founder of comput...",38667
...,...,...,...
38823,Q9047,Section: Writings and publication. At the same...,38823
38824,Q9047,Section: Writings and publication. The ambiti...,38824
38825,Q9047,Section: Selected works. The year given is usu...,38825
38826,Q9047,Section: Collections. Six important collection...,38826


In [40]:
with open("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/test/qrelv2.txt", 'w') as f:
    for line in qrel2:
        f.write(line+"\n")

In [63]:
cleanup_dist_env_and_memory()
del llm

NameError: name 'llm' is not defined

In [64]:
corpus = pd.read_parquet("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/test/corpus.parquet")

In [67]:
for ix, row in corpus_parquet.iterrows():
    print(row)
    break

wiki_id                                                 Q199
text       1  1 (one, also called unit, and unity) is a n...
docid                                                      0
Name: 0, dtype: object


In [ ]:
qrel = pd.read_csv("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/test/qrelv2.txt", sep=" ", names=["qid", "zero", "docid", "label"])

In [68]:
query_parquet

,query,answers,query_id
0,Which team did Attaphol Buspakom play for from...,"[Port F.C, Thailand national football team]",0
18,Which team did the player Attaphol Buspakom be...,"[Pahang FA, Thailand national football team]",1
36,Attaphol Buspakom played for which team from 1...,"[Port F.C, Thailand national football team]",2
54,Which team did the player Attaphol Buspakom be...,"[Pahang FA, Thailand national football team]",3
72,Which team did the player Attaphol Buspakom be...,[Thailand national football team],4
...,...,...,...
121320,What was the working location for Gottfried Wi...,[Hanover],5162
121485,What was the working location for Gottfried Wi...,[Wolfenbüttel],5163
121650,Jörg Widmann became a member of what organizat...,[Akademie der Wissenschaften und der Literatur],5164
121671,Jörg Widmann became a member of what organizat...,[Freie Akademie der Künste Hamburg],5165


In [55]:
qrel['qid'].value_counts().head(10)

qid
1967    11
1206     9
2497     9
977      8
978      8
1750     7
1115     7
1007     7
823      7
986      7
Name: count, dtype: int64

In [318]:
wrong = []
for ix in range(len(qrel)):
    if ix < 80:
        continue
    
    row = qrel.iloc[ix]

    qid = row['qid']
    docid = row['docid']
    
    answer = query_parquet[query_parquet['query_id'] == qid]['answers'].values[0][0]
    wiki_id = corpus_parquet[corpus_parquet['docid'] == docid]['wiki_id'].values[0]
    text = " ".join(corpus_parquet[corpus_parquet['wiki_id'] == wiki_id]['text'].values)

    if not answer in text:
        wrong.append(qid)

# Dataset statistics

## Temporal Nobel Prize

In [18]:
original_train_jsonl = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/train/original/contriever_finetune_train_v3.jsonl", jsonl=True)
original_dev_jsonl = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/train/original/contriever_finetune_eval_v3.jsonl", jsonl=True)

our_train_jsonl = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/train/train_enhanced_temporal_v2.jsonl", jsonl=True)
our_dev_jsonl = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/temporal_nobel_prize/train/dev3.jsonl", jsonl=True)

Reading the json file...
The file is of type: <class 'list'>
The file contains 8060 items.
Reading the json file...
The file is of type: <class 'list'>
The file contains 165 items.
Reading the json file...
The file is of type: <class 'list'>
The file contains 11693 items.
Reading the json file...
The file is of type: <class 'list'>
The file contains 306 items.


### Count positive/negative

In [ ]:
count_question = 0
count_positive = 0
count_negative = 0

query_id = []

for i in range(len(original_train_jsonl)):
    row = original_train_jsonl[i]

    count_question += 1
    count_positive += len(row['positive_ctxs'])
    count_negative += len(row['negative_ctxs'])
    query_id.append(row['question'])

pprint({
    "count_question": count_question,
    "count_positive": count_positive,
    "count_negative": count_negative,
    "len(set(query_id))": len(set(query_id))
})

{'count_negative': 23388,
 'count_positive': 23388,
 'count_question': 8060,
 'len(set(query_id))': 8060}


In [ ]:
count_question = 0
count_positive = 0
count_negative = 0

query_id = []

for i in range(len(original_dev_jsonl)):
    row = original_dev_jsonl[i]

    count_question += 1
    count_positive += len(row['positive_ctxs'])
    count_negative += len(row['negative_ctxs'])
    query_id.append(row['question'])
    
pprint({
    "count_question": count_question,
    "count_positive": count_positive,
    "count_negative": count_negative,
    "len(set(query_id))": len(set(query_id))
})

{'count_negative': 468,
 'count_positive': 468,
 'count_question': 165,
 'len(set(query_id))': 165}


In [21]:
count_question = 0
count_positive = 0
count_negative = 0

query_id = []

for i in range(len(our_train_jsonl)):
    row = our_train_jsonl[i]
    
    query_id.append(row['query_id'])

    count_question += 1
    count_positive += len(row['positive_passages'])
    count_negative += len(row['negative_passages'])
    
pprint({
    "count_question": count_question,
    "count_positive": count_positive,
    "count_negative": count_negative,
    "len(set(query_id))": len(set(query_id))
})

{'count_negative': 60563,
 'count_positive': 64564,
 'count_question': 11693,
 'len(set(query_id))': 5922}


In [27]:
our_dev_jsonl[24]

{'query_id': 10,
 'query': 'Lubbers semi-retired from active politics and became active in the public sector as a non-profit director and served on several and councils on behalf of the government , he also served as a distinguished visiting professor of International relations and Globalization at the Tilburg University and the John F . Kennedy School of Government of the Harvard University in Cambridge , Massachusetts from February 1995 until December 2000 . In November 2000 Lubbers was nominated as the next United Nations High Commissioner for Refugees serving from 1 January 2001 until 20 February 2005 . Following his retirement Lubbers continued to be active public sector and worked as an advocate , lobbyist and activist for Humanitarian , Conservation , Environmentalism , Sustainable development and Climate change issues .',
 'positive_passages': [{'docid': 2302,
   'text': 'What was the position of Ruud Lubbers from 2001 to Feb 2005?'},
  {'docid': 2302,
   'text': 'Ruud Lubbers 

### Count allen relations, temporal query types

In [26]:
from collections import defaultdict
pos_allen_count_dict = defaultdict(int)
pos_query_type_count_dict = defaultdict(int)
neg_allen_count_dict = defaultdict(int)
neg_query_type_count_dict = defaultdict(int)
count = defaultdict(int)

for i in range(len(our_dev_jsonl)):
    row = our_dev_jsonl[i]
    
    for j in row['positive_passages']:
        # pos_allen_count_dict[j['allen_relation']] += 1
        # pos_query_type_count_dict[j['temporal_query_type']] += 1
        try:
            count[j['temporal_query_type']] += 1
            pos_allen_count_dict[j['allen_relation']+"_"+j['temporal_query_type']] += 1
        except:
            print(i, row['query_id'])
        
        # if j['allen_relation'] == "Empty" and j['temporal_query_type'] != "TemporalAnswer":
        #     print("Positive Passage:")
        #     print(row['query'])
        #     print(row['temporal'])
        #     print(j['text'])
        #     print("-----")
        
    for j in row['negative_passages']:
        # neg_allen_count_dict[j['allen_relation']] += 1
        # neg_query_type_count_dict[j['temporal_query_type']] += 1
        try:
            count[j['temporal_query_type']] += 1
            pos_allen_count_dict[j['allen_relation']+"_"+j['temporal_query_type']] += 1
        except:
            print(i, row['query_id'])
        # if j['allen_relation'] == "Empty" and j['temporal_query_type'] != "TemporalAnswer":
        #     print("Negative Passage:")
        #     print(row['query'])
        #     print(row['temporal'])
        #     print(j['text'])
        #     print("-----")
        
pprint(pos_allen_count_dict)
pprint(neg_allen_count_dict)
print(count)
# pprint(pos_query_type_count_dict)
# pprint(neg_query_type_count_dict)

24 10
24 10
24 10
24 10
68 29
68 29
68 29
68 29
71 33
71 33
71 33
71 33
112 49
112 49
112 49
112 49
171 71
171 71
171 71
171 71
195 86
195 86
195 86
195 86
195 86
195 86
195 86
195 86
203 90
203 90
203 90
203 90
204 91
204 91
204 91
204 91
223 102
223 102
223 102
223 102
224 103
224 103
224 103
224 103
224 103
224 103
224 103
224 103
225 104
225 104
225 104
225 104
227 106
227 106
227 106
227 106
227 106
227 106
227 106
227 106
228 107
228 107
228 107
228 107
229 108
229 108
229 108
229 108
245 117
245 117
245 117
245 117
246 118
246 118
246 118
246 118
248 124
248 124
248 124
248 124
249 125
249 125
249 125
249 125
249 125
249 125
249 125
249 125
258 131
258 131
258 131
258 131
268 139
268 139
268 139
268 139
268 139
268 139
268 139
268 139
268 139
268 139
268 139
268 139
269 140
269 140
269 140
269 140
270 141
270 141
270 141
270 141
270 141
270 141
270 141
270 141
270 141
270 141
270 141
270 141
274 145
274 145
274 145
274 145
275 146
275 146
275 146
275 146
290 154
290 154
290 154


In [9]:
count_question = 0
count_positive = 0
count_negative = 0

query_id = []

for i in range(len(our_test_corpus_jsonl)):
    row = our_test_corpus_jsonl[i]
    
    query_id.append(row['query_id'])

    count_question += 1
    count_positive += len(row['positive_passages'])
    count_negative += len(row['negative_passages'])
    
print(count_question, count_positive, count_negative, len(set(query_id)))

306 1630 1520 146


## TimeQA

In [59]:
temp = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/train/original_train.jsonl", jsonl=True)

original_train_jsonl = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/train/train.jsonl", jsonl=True)
original_dev_jsonl = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/train/dev.jsonl", jsonl=True)

our_train_jsonl = read_json("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/development/train.jsonl", jsonl=True)

Reading the json file...
The file is of type: <class 'list'>
The file contains 9166 items.
Reading the json file...
The file is of type: <class 'list'>
The file contains 5143 items.
Reading the json file...
The file is of type: <class 'list'>
The file contains 826 items.
Reading the json file...
The file is of type: <class 'list'>
The file contains 8149 items.


In [61]:
count_question = 0
count_positive = 0
count_negative = 0

query_id = []

for i in range(len(original_dev_jsonl)):
    row = original_dev_jsonl[i]

    count_question += 1
    count_positive += len(row.get('positive_passages', []))
    count_negative += len(row.get('negative_passages', []))
    query_id.append(row['positive_passages'][0]['docid'])

pprint({
    "count_question": count_question,
    "count_positive": count_positive,
    "count_negative": count_negative,
    "len(set(query_id))": len(set(query_id))
})

{'count_negative': 0,
 'count_positive': 3162,
 'count_question': 826,
 'len(set(query_id))': 622}


### Test corpus/queries/qrel

In [64]:
test_corpus_parquet = pd.read_parquet("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/test/corpus.parquet")
test_queries_parquet = pd.read_parquet("/home/thuy0050/mg61_scratch2/thuy0050/data/third_work/temporal/time_sensitive_qa/test/query.parquet")

In [68]:
test_corpus_parquet

,wiki_id,text,docid
0,Q199,"1 1 (one, also called unit, and unity) is a n...",0
1,Q199,The fundamental mathematical property of 1 is ...,1
2,Q199,"It commonly denotes the first, leading or top ...",2
3,Q199,"Section: Etymology. The word ""one"" can be used...",3
4,Q199,Section: Etymology. Compare the Proto-Indo-Eu...,4
...,...,...,...
109177,Q384172,"Section: Celtic. Zaluska, an used substitute i...",109177
109178,Q384172,Section: Celtic. He left Celtic in June 2015. ),109178
109179,Q384172,"Section: Darmstadt 98. On 31 August 2015, Zału...",109179
109180,Q384172,"Section: Pogoń Szczecin. On 9 June 2017, he si...",109180


In [66]:
test_queries_parquet

,query,answers,query_id
0,Which team did Attaphol Buspakom play for from...,"[Port F.C, Thailand national football team]",0
18,Which team did the player Attaphol Buspakom be...,"[Pahang FA, Thailand national football team]",1
36,Attaphol Buspakom played for which team from 1...,"[Port F.C, Thailand national football team]",2
54,Which team did the player Attaphol Buspakom be...,"[Pahang FA, Thailand national football team]",3
72,Which team did the player Attaphol Buspakom be...,[Thailand national football team],4
...,...,...,...
121320,What was the working location for Gottfried Wi...,[Hanover],5162
121485,What was the working location for Gottfried Wi...,[Wolfenbüttel],5163
121650,Jörg Widmann became a member of what organizat...,[Akademie der Wissenschaften und der Literatur],5164
121671,Jörg Widmann became a member of what organizat...,[Freie Akademie der Künste Hamburg],5165


In [70]:
from collections import defaultdict
pos_allen_count_dict = defaultdict(int)
pos_query_type_count_dict = defaultdict(int)
neg_allen_count_dict = defaultdict(int)
neg_query_type_count_dict = defaultdict(int)
count = defaultdict(int)

for i in range(len(our_train_jsonl)):
    row = our_train_jsonl[i]
    
    for j in row['positive_passages']:
        # pos_allen_count_dict[j['allen_relation']] += 1
        # pos_query_type_count_dict[j['temporal_query_type']] += 1
        try:
            count[j['temporal_query_type']] += 1
            pos_allen_count_dict[j['allen_relation']+"_"+j['temporal_query_type']] += 1
        except:
            print(i, row['query_id'])
        
        # if j['allen_relation'] == "Empty" and j['temporal_query_type'] != "TemporalAnswer":
        #     print("Positive Passage:")
        #     print(row['query'])
        #     print(row['temporal'])
        #     print(j['text'])
        #     print("-----")
        
    for j in row['negative_passages']:
        # neg_allen_count_dict[j['allen_relation']] += 1
        # neg_query_type_count_dict[j['temporal_query_type']] += 1
        try:
            count[j['temporal_query_type']] += 1
            pos_allen_count_dict[j['allen_relation']+"_"+j['temporal_query_type']] += 1
        except:
            print(i, row['query_id'])
        # if j['allen_relation'] == "Empty" and j['temporal_query_type'] != "TemporalAnswer":
        #     print("Negative Passage:")
        #     print(row['query'])
        #     print(row['temporal'])
        #     print(j['text'])
        #     print("-----")
        
pprint(pos_allen_count_dict)
pprint(neg_allen_count_dict)
print(count)
# pprint(pos_query_type_count_dict)
# pprint(neg_query_type_count_dict)

defaultdict(<class 'int'>,
            {'After_Explicit': 24224,
             'After_Implicit': 320,
             'Before_Explicit': 28188,
             'Before_Implicit': 235,
             'Contains_Explicit': 1024,
             'Contains_Implicit': 1,
             'During_Explicit': 12081,
             'During_Implicit': 134,
             'Empty_Explicit': 19,
             'Empty_Implicit': 116,
             'Empty_TemporalAnswer': 8514,
             'Equals_Explicit': 6551,
             'Equals_Implicit': 54,
             'FinishedBy_Explicit': 2,
             'Finishes_Explicit': 121,
             'Finishes_Implicit': 2,
             'Meets_Explicit': 52,
             'Meets_Implicit': 5,
             'MetBy_Explicit': 23,
             'MetBy_Implicit': 12,
             'OverlappedBy_Explicit': 52,
             'Overlaps_Explicit': 3922,
             'Overlaps_Implicit': 21,
             'StartedBy_Explicit': 10,
             'StartedBy_Implicit': 1,
             'Starts_Explicit':